# TDQUAL: Experimental Evaluation on Machine Learning Tasks

In this notebook, we explore the use of the algorithm developed in the TDQUAL project to score and rank training subsets based on the distances computed from the matching diagram. Using the SNIPS dataset, a widely used benchmark for intent classification in NLP, we analyze how the algorithm identifies potentially informative subsets that could be used to reduce training data requirements.

For an analogous analysis based on a different data embedding, see the companion notebook `TDQUAL_SNIPS_768_2D.ipynb`

## Installations and Imports

If you are running this notebook in Colab, you only need to install the `skorch` library.

In [ ]:
#!pip install skorch

To begin, we import the necessary libraries and load the model that will generate sentence embeddings.

For this experiment, we use the 384-dimensional SBERT model `all-MiniLM-L6-v2`.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
import random
import inspect
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import pandas as pd
from pandas.plotting import andrews_curves
from mpl_toolkits.mplot3d import Axes3D
import plotly.express as px

import umap

from sklearn import datasets
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score

import torch
import torch.nn as nn

from skorch import NeuralNetClassifier
from skorch.callbacks import EpochScoring
from skorch.callbacks import EarlyStopping

import scipy.spatial.distance as dist
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.sparse import coo_matrix
from scipy.stats import norm

## Training Manager

To manage training runs and results, we use a Training Manager capable of handling machine learning experiments with both scikit-learn and skorch models.

In [ ]:
class TrainingManager:

    def __init__(self):
        self.history = {}

    def create_experiment(
        self,
        name,
        model_class,
        data,
        max_runs,
        params=None,
        framework="sklearn"
    ):
        """
        data format:

        {
            "train": (X_train, y_train),
            "val": (X_val, y_val),  # opcional
            "test": (X_test, y_test)
        }
        """

        if name in self.history:
            raise ValueError(f"The experiment '{name}' already exists")

        self.history[name] = {
            "model_class": model_class,
            "params": params or {},
            "framework": framework,
            "data": data,
            "max_runs": max_runs,
            "runs": []
        }

    def run_experiment(
        self,
        experiment_name,
        seed=0,
        metrics_fn=None,
        model_metrics_fn=None
    ):

        experiment = self.history[experiment_name]
        framework = experiment["framework"]

        train_data = experiment["data"]["train"]
        test_data = experiment["data"]["test"]

        X_train, y_train = train_data
        X_test, y_test = test_data

        val_data = experiment["data"].get("val")

        metrics_fn = metrics_fn or {}
        model_metrics_fn = model_metrics_fn or {}

        model_class = experiment["model_class"]

        for i in range(experiment["max_runs"]):

            current_seed = seed + i


            # Reproducibility

            random.seed(current_seed)
            np.random.seed(current_seed)
            torch.manual_seed(current_seed)

            # Params

            params = experiment["params"].copy()

            # random_state for sk_learn

            sig = inspect.signature(model_class.__init__)
            if "random_state" in sig.parameters:
                params["random_state"] = current_seed

            model = model_class(**params)


            # Validation for skorch

            if framework == "skorch" and val_data is not None:

                from skorch.helper import predefined_split
                from skorch.dataset import Dataset

                X_val, y_val = val_data

                val_dataset = Dataset(X_val, y_val)

                model.set_params(
                    train_split=predefined_split(val_dataset)
                )

            # Train

            model.fit(X_train, y_train)

            # Predict

            preds = model.predict(X_test)

            # Metrics sklearn

            metrics = {
                name: fn(y_test, preds)
                for name, fn in (metrics_fn or {}).items()
            }

            # Metrics skorch

            model_metrics = {
                name: fn(model)
                for name, fn in (model_metrics_fn or {}).items()
            }


            # History

            training_history = getattr(model, "history", None)

            run_data = {
                "run_id": i + 1,
                "seed": current_seed,
                "metrics": metrics,
                "model_metrics": model_metrics,
                "training_history": training_history,
                "model": model
            }

            experiment["runs"].append(run_data)


    def clear_experiment(self, experiment_name):

        if experiment_name not in self.history:
            raise ValueError(
                f"The experiment '{experiment_name}' already exists"
            )

        self.history[experiment_name]["runs"] = []

    def get_statistics(self, experiment_name):

        runs = self.history[experiment_name]["runs"]

        if len(runs) == 0:
            return {}

        stats = {}

        metric_names = runs[0]["metrics"].keys()

        for metric in metric_names:

            values = [r["metrics"][metric] for r in runs]

            stats[metric] = {
                "mean": float(np.mean(values)),
                "std": float(np.std(values)),
                "min": float(np.min(values)),
                "max": float(np.max(values))
            }

        return stats

    def plot_loss_and_validation(
        self,
        experiment_names,
        figsize=(10, 6),
        alpha=0.3,
        mean_curve=False
    ):

        import numpy as np
        import matplotlib.pyplot as plt


        if isinstance(experiment_names, str):
            experiment_names = [experiment_names]

        colors = [
            ("red", "blue"),
            ("darkorange", "green"),
            ("purple", "cyan"),
            ("brown", "magenta"),
            ("black", "gray")
        ]

        fig, ax1 = plt.subplots(figsize=figsize)
        ax2 = ax1.twinx()

        for exp_idx, experiment_name in enumerate(experiment_names):

            runs = self.history[experiment_name]["runs"]

            loss_curves = []
            val_curves = []

            loss_color, val_color = colors[exp_idx % len(colors)]

            for run in runs:

                history = run.get("training_history")

                if history is not None:

                    loss = []
                    val = []

                    for row in history:

                        if "train_loss" in row:
                            loss.append(row["train_loss"])

                        if "valid_acc" in row:
                            val.append(row["valid_acc"])

                    if len(loss) > 0:

                        loss_curves.append(loss)

                        ax1.plot(
                            loss,
                            color=loss_color,
                            alpha=alpha,
                            linewidth=1
                        )

                    if len(val) > 0:

                        val_curves.append(val)

                        ax2.plot(
                            val,
                            color=val_color,
                            alpha=alpha,
                            linewidth=1
                        )

                else:

                    loss = run["model_metrics"].get("loss_curve")
                    val = run["model_metrics"].get("validation_scores")

                    if loss is None:
                        continue

                    loss_curves.append(loss)

                    ax1.plot(
                        loss,
                        color=loss_color,
                        alpha=alpha,
                        linewidth=1
                    )

                    if val is not None:

                        val_curves.append(val)

                        ax2.plot(
                            val,
                            color=val_color,
                            alpha=alpha,
                            linewidth=1
                        )

            if mean_curve and len(loss_curves) > 0:

                min_loss = min(len(c) for c in loss_curves)

                mean_loss = np.mean(
                    np.array([c[:min_loss] for c in loss_curves]),
                    axis=0
                )

                ax1.plot(
                    mean_loss,
                    color=loss_color,
                    linewidth=3,
                    label=f"{experiment_name} Mean Loss"
                )

                if len(val_curves) > 0:

                    min_val = min(len(c) for c in val_curves)

                    mean_val = np.mean(
                        np.array([c[:min_val] for c in val_curves]),
                        axis=0
                    )

                    ax2.plot(
                        mean_val,
                        color=val_color,
                        linewidth=3,
                        linestyle="--",
                        label=f"{experiment_name} Mean Validation"
                    )

        title = " | ".join(experiment_names)

        ax1.set_title(f"Loss & Validation - {title}")

        ax1.set_xlabel("Epoch")

        ax1.set_ylabel("Loss")

        ax2.set_ylabel("Validation")

        ax1.grid(True)

        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()

        ax1.legend(
            lines1 + lines2,
            labels1 + labels2,
            loc="best"
        )

        plt.show()

## TDQUAL Functions

To avoid dependency on the project library, we load only the strictly necessary functions.

In [ ]:
def subsetZ_stratified(Z, y, size, random_state=None):
    """Selects a stratified random subset of size `size` from the input data, preserving the class distribution of the target labels.

    Returns a stratified subset `X` of `Z` and the corresponding indices with respect to the full dataset `Z`.
    """
    rng = np.random.default_rng(random_state)

    classes, counts = np.unique(y, return_counts=True)
    proportions = counts / len(y)

    selected_idx = []

    for c, p in zip(classes, proportions):
        class_idx = np.where(y == c)[0]
        n_c = int(round(size * p))

        chosen = rng.choice(class_idx, size=n_c, replace=False)
        selected_idx.append(chosen)

    selected_idx = np.concatenate(selected_idx)
    rng.shuffle(selected_idx)

    X = Z[selected_idx]
    y_sub = y[selected_idx]

    return X, y_sub, selected_idx

def reorder_subset(Z, labels, idx, label=None):
    """
    Reorders a dataset by moving a selected subset of indices to the front while keeping the remaining samples in their original relative order.
    Optionally filters the dataset by a specific label before performing the reordering, updating the indices accordingly.

    Returns the reordered dataset, reordered labels, and the reordered subset corresponding to the selected indices.
    """
    idx = np.array(idx)

    if label is not None:

        label_mask = labels == label

        valid_indices = np.where(label_mask)[0]

        Z = Z[label_mask]
        labels = labels[label_mask]

        idx = idx[np.isin(idx, valid_indices)]

        mapping = {
            old: new
            for new, old in enumerate(valid_indices)
        }

        idx = np.array([mapping[i] for i in idx])

    mask = np.ones(len(Z), dtype=bool)
    mask[idx] = False

    Z_ord = np.vstack((Z[idx], Z[mask]))
    labels_ord = np.concatenate((labels[idx], labels[mask]))

    X_ord = Z_ord[:len(idx)]

    idx_new = np.arange(len(idx))

    return Z_ord, labels_ord, X_ord


def mst_edge_filtration(points, is_dist=False):
    """Returns the edges and filtration values for the Euclidean minimum spanning tree
    of a given point sample.
    This is a wrapper for the scipy minimum_spanning_tree function.
    """
    if is_dist:
      mst = minimum_spanning_tree(points)
    else:
      d = dist.pdist(points)
      n = len(points)
      rows, cols = np.triu_indices(n, k=1)

      graph = coo_matrix(
          (d, (rows, cols)),
          shape=(n, n)
      )

      graph = graph + graph.T

      mst = minimum_spanning_tree(graph)

    return read_csr_matrix(mst)

def read_csr_matrix(cs_matrix):
    """Function to read output from minimum_spanning_tree and prepare it as a list of
    filtration values (in order) together with an array with the corresponding pairs.
    """
    rows, cols = cs_matrix.nonzero()

    weights = cs_matrix.data

    sort_idx = np.argsort(weights)

    edges = np.column_stack((rows, cols))[sort_idx].astype(np.int64)
    filtration_list = weights[sort_idx]

    return filtration_list.tolist(), edges

def compute_tmt_pairs_history(
    filtration_list,
    edges_arr,
    tolerance=1e-8
):
    """Computes the complete merge history of a Topological Merger Tree (TMT) from a filtration and a list of edges.
    Returns the sequence of merged component pairs, the corresponding merge sizes, the full size history, and the filtration values at which merges occur.
    """
    n = edges_arr.max() + 1

    # labels de componentes
    C = np.arange(n, dtype=np.int64)

    # tamaños de componentes
    sizes = np.ones(n, dtype=np.int64)

    tmt_pairs = []
    merge_sizes = []

    # historial COMPLETO
    size_history = [sizes.copy()]

    # estado inicial
    filtration_history = [0.0]

    E_b = []

    for i, (b, edge) in enumerate(zip(filtration_list, edges_arr)):

        E_b.append(edge)

        # acumular mismo nivel
        if (
            i < len(filtration_list) - 1
            and abs(b - filtration_list[i + 1]) < tolerance
        ):
            continue

        while E_b:

            mins = np.array([
                min(C[u], C[v])
                for u, v in E_b
            ])

            idx = np.argmin(mins)

            u, v = E_b.pop(idx)

            cu, cv = C[u], C[v]

            M = max(cu, cv)
            m = min(cu, cv)

            # ya conectados
            if M == m:
                continue

            # tamaños antes del merge
            size_M = sizes[M]
            size_m = sizes[m]

            merge_sizes.append([size_M, size_m])

            # merge TMT
            tmt_pairs.append([M, m])

            # merge labels
            C[C == M] = m

            # actualizar tamaños
            sizes[m] += sizes[M]
            sizes[M] = 0

            # snapshot tras merge
            size_history.append(sizes.copy())

            filtration_history.append(b)

    return (
        np.array(tmt_pairs, dtype=np.int64),
        np.array(merge_sizes, dtype=np.int64),
        np.array(size_history, dtype=np.int64),
        np.array(filtration_history)
    )

def get_inclusion_matrix(pairs_arr_X, pairs_arr_Z, subset_indices=[]):
    """ Given two pairs of arrays with the vertex merge pairs, this function returns the associated inclusion matrix.
    From the point of view of minimum spanning trees, the output matrix columns can be interpreted as the minimum paths that are needed to
    go through in MST(Z) in order to connect the endpoints from an edge in MST(X)
    """
    # If subset indices are not specified, we assume that the indices of vertices from S correspond to the first #X vertices from Z
    if (len(subset_indices)==0):
        subset_indices = list(range(pairs_arr_X.shape[0]+1))
    pivot2column = [-1] + np.argsort(np.max(pairs_arr_Z, axis=1)).tolist()
    inclusion_matrix = []
    for col_X in pairs_arr_X:
        col_X = [subset_indices[i] for i in col_X]
        col_M = []
        while(len(col_X)>0):
            piv = np.max(col_X)
            col_M.append(pivot2column[piv])
            col_X = add_columns_mod_2(col_X, pairs_arr_Z[pivot2column[piv]])
        # end reducing column X
        col_M.sort()
        inclusion_matrix.append(col_M)
    return inclusion_matrix

def add_columns_mod_2(col1, col2):
    """ Given two lists of integers, which are sparse representations of a pair of vectors in Z mod 2, this funciton adds them and
    returns the result in the same input format.
    """
    return list(set(col1) ^ set(col2)) #La operación ^ entre sets da los elementos que están en cada conjunto pero no en ambos


def get_inclusion_matrix_pivots(matrix_list, num_rows):
    """ Returns the pivots of a matrix given in list format"""
    pivots = []
    reduced_matrix = []
    pivot2column = np.full(num_rows, -1, dtype=int)
    for i, column in enumerate(matrix_list):
        reduce_column = list(column)
        piv = get_pivot(reduce_column)
        while(pivot2column[piv]>-1):
            reduce_column = add_columns_mod_2(reduce_column, reduced_matrix[pivot2column[piv]])
            piv = get_pivot(reduce_column)
            # we assume that columns are never reduced to the 0 column
        pivots.append(piv)
        reduced_matrix.append(reduce_column)
        pivot2column[piv] = i
    # end getting pivots
    return pivots

def get_pivot(nonempty_list):
    "given a non-empty list of integers, returns the pivot. GIves error otherwise."
    return int(np.max(nonempty_list))

def compute_Mf_0(X, Z, is_dist=False, track_reductions=False):
    """Compute the matching induced by the inclusion X --> Z given by sending points from X, in order, to the first points from Z.
    We assume that coordinates of X and Z are given as numpy arrays, where the number of rows from Z is greater or equal to X.
    If is_dist is True, we assume that X and Z are given as distance matrices.

    This returns a pair of lists with endpoints of intervals, that are representations of both barcodes, together with a matching between such lists given by a list of indices.
    """
    filtration_list_X, pairs_arr_X = mst_edge_filtration(X, is_dist=is_dist) # MST(X)
    filtration_list_Z, pairs_arr_Z = mst_edge_filtration(Z, is_dist=is_dist) # MST(Z)
    TMT_X_pairs, merge_sizes_X, _, _  = compute_tmt_pairs_history(filtration_list_X, pairs_arr_X)
    TMT_Z_pairs, merge_sizes_Z, _, _ = compute_tmt_pairs_history(filtration_list_Z, pairs_arr_Z)
    indices_X_Z = np.max(TMT_Z_pairs, axis=1)<X.shape[0]
    TMT_X_Z_pairs = TMT_Z_pairs[indices_X_Z]
    merge_sizes_X_Z = merge_sizes_Z[indices_X_Z]
    indices_X_Z = np.nonzero(indices_X_Z)[0]
    FX = get_inclusion_matrix(TMT_X_pairs, TMT_X_Z_pairs) # Associated matrix
    if track_reductions:
      matchingX, reductions = get_inclusion_matrix_pivots_tracking_reductions(FX, Z.shape[0]) # Matching in TMT_X_Z
      matching =[indices_X_Z[i] for i in matchingX] # Matching in all TMT_Z
      return filtration_list_X,TMT_X_pairs,merge_sizes_X ,filtration_list_Z, TMT_Z_pairs, merge_sizes_Z, TMT_X_Z_pairs, merge_sizes_X_Z, matching, reductions
    else:
      matchingX = get_inclusion_matrix_pivots(FX, Z.shape[0]) # Matching in TMT_X_Z
      matching =[indices_X_Z[i] for i in matchingX] # Matching in all TMT_Z
      return filtration_list_X,TMT_X_pairs,merge_sizes_X ,filtration_list_Z, TMT_Z_pairs, merge_sizes_Z, TMT_X_Z_pairs, merge_sizes_X_Z, matching

def compute_matching_diagram(filt_X, filt_Z, matching, tol=1e-5):
    """Computes the matching diagram between `filt_X` and `filt_Z`, grouping repeated values within a tolerance.
    Returns the matching pairs between the two filtrations and their corresponding counts.
    """
    filt_X = np.asarray(filt_X)
    filt_Z = np.asarray(filt_Z)
    matching = np.asarray(matching)

    # -----------------------------------------
    # Matched pairs
    # -----------------------------------------

    pairs = np.column_stack((
        filt_X,
        filt_Z[matching]
    ))

    # Reverse to mimic pop()
    pairs = pairs[::-1]

    diff = np.abs(np.diff(pairs, axis=0))

    same = np.all(diff < tol, axis=1)

    group_start = np.concatenate((
        [True],
        ~same
    ))

    idx = np.flatnonzero(group_start)

    unique_pairs = pairs[idx]

    counts = np.diff(np.append(idx, len(pairs)))

    # -----------------------------------------
    # Unmatched points
    # -----------------------------------------

    mask = np.ones(len(filt_Z), dtype=bool)
    mask[matching] = False

    unmatched = filt_Z[mask][::-1]

    if unmatched.size > 0:

        diff_u = np.abs(np.diff(unmatched))

        same_u = diff_u < tol

        group_start_u = np.concatenate((
            [True],
            ~same_u
        ))

        idx_u = np.flatnonzero(group_start_u)

        unique_unmatched = unmatched[idx_u]

        counts_u = np.diff(
            np.append(idx_u, len(unmatched))
        )

        unmatched_pairs = np.column_stack((
            np.full(len(unique_unmatched), np.inf),
            unique_unmatched
        ))

        unique_pairs = np.vstack((
            unique_pairs,
            unmatched_pairs
        ))

        counts = np.concatenate((
            counts,
            counts_u
        ))

    return unique_pairs, counts


def mixup_percentage_distance(points, mult):
    """Computes a mixup percentage distance between paired points.
    """
    mult = np.asarray(mult)
    _points = points[points[:,0] != np.inf]
    _mult = mult[points[:,0] != np.inf]

    x = _points[:,0]
    y = _points[:,1]

    return np.sum(_mult * np.abs((x - y)/y)).item()

def induced_matching_distance_L1(points, mult, matching_type = None):
    """Computes a weighted L1 distance for a set of matched or unmatched point pairs.
    If `matching_type` is 'matched', the distance is computed over finite pairs only.
    If it is 'unmatched', the distance is computed over pairs with infinite first coordinate, treating infinity as zero.
    If `matching_type` is None, the distance is computed over all points using the same convention for infinite values.
    """
    mult = np.asarray(mult)

    if matching_type == 'matched':

      _points = points[points[:,0] != np.inf]
      _mult = mult[points[:,0] != np.inf]

      x = _points[:,0]
      y = _points[:,1]

      return np.sum(_mult * np.abs((x - y))).item()

    elif matching_type == 'unmatched':

      _points = points[points[:,0] == np.inf]
      _mult = mult[points[:,0] == np.inf]

      x = np.where(np.isinf(_points[:, 0]), 0, _points[:, 0])
      y = _points[:,1]

      return np.sum(_mult * np.abs((x - y))).item()

    else:
      x = np.where(np.isinf(points[:, 0]), 0, points[:, 0])
      y = points[:, 1]

      return np.sum(mult * np.abs(x - y)).item()


def matching_distance_score(subsets, Z, y, label, is_dist=False):
  """
  Returns the L1 distance values for each subset in the input list.
  """
  d_l1_matches = []
  d_l1_unmatches = []
  d_l1_mixup = []

  if is_dist:
    for i in range(len(subsets)):
      Zi, yi, Xi = reorder_subset(Z, y, subsets[i][2], label=label)

      similarity_Zi = Zi @ Zi.T
      distance_Zi = 1 - similarity_Zi

      similarity_Xi = Xi @ Xi.T
      distance_Xi = 1 - similarity_Xi

      filt_Xi, _,_, filt_Zi, _,_, _, _, matchingi = compute_Mf_0(distance_Xi,distance_Zi,is_dist=is_dist)
      D_fi, mults = compute_matching_diagram(filt_Xi, filt_Zi,matchingi)

      d_l1_matches.append(induced_matching_distance_L1(D_fi, mults, matching_type='matched'))
      d_l1_unmatches.append(induced_matching_distance_L1(D_fi, mults, matching_type='unmatched'))
      d_l1_mixup.append(mixup_percentage_distance(D_fi, mults))


  else:
    for i in range(len(subsets)):
      Zi, yi, Xi = reorder_subset(Z, y, subsets[i][2], label=label)

      filt_Xi, _,_, filt_Zi, _,_, _, _, matchingi = compute_Mf_0(Xi,Zi,is_dist=is_dist)
      D_fi, mults = compute_matching_diagram(filt_Xi, filt_Zi,matchingi)

      d_l1_matches.append(induced_matching_distance_L1(D_fi, mults, matching_type='matched'))
      d_l1_unmatches.append(induced_matching_distance_L1(D_fi, mults, matching_type='unmatched'))
      d_l1_mixup.append(mixup_percentage_distance(D_fi, mults))

  return d_l1_matches, d_l1_unmatches, d_l1_mixup

## Data Loading and Preprocessing (SNIPS)

We load the dataset from the `snips_only_intent.csv` file. A sample of the data and a summary of the number of sentences per class are shown in the following cells.

In [ ]:
df = pd.read_csv("snips_only_intent.csv").iloc[:, 1:]
df.sample(10)

In [ ]:
print(f'Shape: {df.shape}')
print(f'NAs in DF: {df.isna().any().any()}\n')
print(df['labels'].value_counts())

We transform the sentences into 384-dimensional vectors using the model loaded at the beginning of the experiment. The SBERT model encodes each full sentence into an embedding that captures semantic similarity.

In [ ]:
embeddings = model.encode(df['sentences'].tolist(), normalize_embeddings=True)
target = np.array(df['labels'].tolist())

In the next cell, we apply the remaining transformations to the dataset, as well as its split into training, validation, and test sets. Specifically, we apply:

1. Label encoding to the target labels.

2. Dimensionality reduction of the embeddings to 2 dimensions using UMAP.

We can visualize the training set representation in the following figure.

In [ ]:
SEED = 42

data = embeddings.copy()

Z_train, Z_val_test, y_train, y_val_test = train_test_split(
    data, target,
    test_size=0.3,
    random_state=SEED,
    shuffle=True,
    stratify=target
)

Z_val, Z_test, y_val, y_test = train_test_split(
    Z_val_test, y_val_test,
    test_size=0.5,
    random_state=SEED+1,
    shuffle=True,
    stratify=y_val_test
)

le = LabelEncoder()

y_train_cat = le.fit_transform(y_train)
y_val_cat   = le.transform(y_val)
y_test_cat  = le.transform(y_test)

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=42
)

Z_train = reducer.fit_transform(Z_train)
Z_val = reducer.transform(Z_val)
Z_test = reducer.transform(Z_test)

n_classes = 7


print(f'Global:')
print(f'Train size: {len(y_train_cat)}')
print(f'Validation size: {len(y_val_cat)}')
print(f'Test size: {len(y_test_cat)}')

print(f'\nSize by classes')
for i in range(n_classes):
  print(f'Class {i}: Train: {np.sum(y_train_cat==i)}, Validation: {np.sum(y_val_cat==i)}, Test: {np.sum(y_test_cat==i)}')


sns.scatterplot(
    x=Z_train[:, 0],
    y=Z_train[:, 1],
    hue=y_train_cat,
    palette='tab10',
    s=20,
    legend=False
)

plt.title('SNIPS 384 - UMAP (2D)')
plt.show()

## Trainings

### Full dataset $Z$

We first define the architecture of the neural network used in the experiment. It has a structure $2 \rightarrow 256 \rightarrow 128 \rightarrow 64 \rightarrow C$, with ReLU activation in the hidden layers.

It is possible to control whether we train and compare the results using the full training set via the `TRAIN_COMPLETE` variable, although this is not the focus of this demonstration. By default, all trainings in this notebook are repeated 10 times using different random seeds. Since we aim to evaluate training performance, all reported metrics are computed on the validation set.

In [ ]:
tm_snips_e = TrainingManager()

TRAIN_COMPLETE = False

class MLPTDQUAL(nn.Module):

    def __init__(self, input_dim):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_dim, 256),
            nn.ReLU(),

            nn.Linear(256, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, n_classes)
        )

    def forward(self, x):

        return self.network(x)

datos_f32_complete = {
    "train": (
        Z_train.astype(np.float32),
        y_train_cat.astype(np.int64)
    ),
    "val": (
        Z_val.astype(np.float32),
        y_val_cat.astype(np.int64)
    ),
    "test": (
        Z_val.astype(np.float32),
        y_val_cat.astype(np.int64)
    )
}

if TRAIN_COMPLETE:
  tm_snips_e.create_experiment(
      name="FCNN_complete",
      framework="skorch",
      model_class=NeuralNetClassifier,
      data=datos_f32_complete,
      max_runs=25,
      params={
          "module": MLPTDQUAL,
          "module__input_dim": datos_f32_complete['train'][0].shape[1],
          "max_epochs": 100,
          "lr": 0.001,
          "batch_size": 64,
          "optimizer": torch.optim.Adam,
          "criterion": nn.CrossEntropyLoss,
          "callbacks": [
              EpochScoring("accuracy", name="valid_acc", lower_is_better=False),
              EarlyStopping(monitor="valid_loss", patience=10)
          ],
          "verbose": 0
      }
  )

  tm_snips_e.run_experiment(
      "FCNN_complete",
      metrics_fn={
          "accuracy": accuracy_score,
          "f1": lambda y_true, y_pred:f1_score(y_true, y_pred, average="micro")
      },
      model_metrics_fn={
          "best_train_loss": lambda m: min(row["train_loss"] for row in m.history),
          "best_valid_loss": lambda m: min(row["valid_loss"] for row in m.history),
          "best_valid_acc": lambda m: max(row["valid_acc"] for row in m.history),
          "epochs": lambda m: len(m.history)
      }
  )

### Subsets $X$

We analyze 10 randomly selected subsets, each containing 100 sentences.

In [ ]:
SEEDS = np.array([12,58,64,128,150,175,200,322,456,790])
size = 100

subsets = [subsetZ_stratified(Z_train,y_train_cat,size, random_state=_seed) for _seed in SEEDS]

print(f'X size: {len(subsets[0][0])}')
print(f'Shape: {subsets[0][0].shape}')
print(f'Counts: {np.unique_counts(subsets[0][1])}')

We can observe that only a small number of sentences appear in more than one subset.

In [ ]:
idx_sets = [set(idx) for _, _, idx in subsets]

n = len(subsets)
common = np.zeros((n, n), dtype=int)

for i in range(n):
    for j in range(i, n):
        n_common = len(idx_sets[i] & idx_sets[j])
        common[i, j] = n_common
        common[j, i] = n_common

sns.heatmap(common, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Subset")
plt.ylabel("Subset")
plt.show()

Next, we perform 10 independent training runs for each subset and visualize the results.

In [ ]:
tm_snips_100 = TrainingManager()

if TRAIN_COMPLETE:
  for experiment in [x for x in tm_snips_e.history.keys()]:
    tm_snips_100.history[experiment] = tm_snips_e.history[experiment]

for subset in range(len(subsets)):

  datos_f32 = {
      "train": (
          subsets[subset][0].astype(np.float32),
          subsets[subset][1].astype(np.int64)
      ),
      "val": (
          Z_val.astype(np.float32),
          y_val_cat.astype(np.int64)
      ),
      "test": (
          Z_val.astype(np.float32),
          y_val_cat.astype(np.int64)
      )
  }


  tm_snips_100.create_experiment(
      name="FCNN_" + str(subset),
      framework="skorch",
      model_class=NeuralNetClassifier,
      data=datos_f32,
      max_runs=25,
      params={
          "module": MLPTDQUAL,
          "module__input_dim": datos_f32['train'][0].shape[1],
          "max_epochs": 100,
          "lr": 0.001,
          "batch_size": 64,
          "optimizer": torch.optim.Adam,
          "criterion": nn.CrossEntropyLoss,
          "callbacks": [
              EpochScoring("accuracy", name="valid_acc", lower_is_better=False),
              EarlyStopping(monitor="valid_loss", patience=10)
          ],
          "verbose": 0
      }
  )

  tm_snips_100.run_experiment(
      "FCNN_" + str(subset),
      metrics_fn={
          "accuracy": accuracy_score,
          "f1": lambda y_true, y_pred:f1_score(y_true, y_pred, average="micro")
      },
      model_metrics_fn={
          "best_train_loss": lambda m: min(row["train_loss"] for row in m.history),
          "best_valid_loss": lambda m: min(row["valid_loss"] for row in m.history),
          "best_valid_acc": lambda m: max(row["valid_acc"] for row in m.history),
          "epochs": lambda m: len(m.history)
      }
  )

In [ ]:
experiment_models = ['FCNN']

for model in experiment_models:
  print(f'----------------{model.upper()}------------------------')
  for experiment in [elemento for elemento in tm_snips_100.history.keys() if (model) in elemento]:
    if experiment[-1] == 'e':
      print(f'\033[34mSubset {experiment[-1]}: {tm_snips_100.get_statistics(experiment)['accuracy']}\033[0m')
    else:
        print(f'Subset {experiment[-1]}: {tm_snips_100.get_statistics(experiment)['accuracy']}')

Now that we have the results, we only need to identify which subsets correspond to the extreme values of the distances: Total Mixup, Total Unmatched, and Total Mixup Percentage.

In [ ]:
scores_0 = matching_distance_score(subsets, Z_train, y_train_cat, label=None)
print(f'---Max---')
print(f'{np.argmax(scores_0, axis=1)} ({np.max(scores_0, axis=1)})')
print(f'---Min---')
print(f'{np.argmin(scores_0, axis=1)} ({np.min(scores_0, axis=1)})')

Max_match_0, Max_unmatch_0, Max_mixup_0 = np.argmax(scores_0, axis=1)
Min_match_0, Min_unmatch_0, Min_mixup_0 = np.argmin(scores_0, axis=1)

### Results

Finally, we present the results in a comparative plot of Gaussian distributions based on the mean and standard deviation.

In [ ]:
rows = []

for model in experiment_models:

    for experiment in tm_snips_100.history:

        if model not in experiment:
            continue

        runs = tm_snips_100.history[experiment]["runs"]

        if experiment.endswith("complete"):
            subset = "complete"
        elif experiment.endswith(str(Max_match_0)):
            subset = str(Max_match_0) + ' - TM (Max) & TMP (Max)'
        elif experiment.endswith(str(Max_unmatch_0)):
            subset = str(Max_unmatch_0) + ' - TU (Max)'
        elif experiment.endswith(str(Min_match_0)):
            subset = str(Min_match_0) + ' - TM (Min)'
        elif experiment.endswith(str(Min_unmatch_0)):
            subset = str(Min_unmatch_0) + ' - TU (Min)'
        elif experiment.endswith(str(Max_mixup_0)):
            subset = str(Max_mixup_0) + ' - TMP (Max)'
        elif experiment.endswith(str(Min_mixup_0)):
            subset = str(Min_mixup_0) + ' - TMP (Min)'
        else:
            subset = experiment[-1]

        for run in runs:

            rows.append({
                "model": model,
                "subset": subset,
                "f1": run["metrics"]["f1"],
                "accuracy": run["metrics"]["accuracy"]
            })

df = pd.DataFrame(rows)

if not TRAIN_COMPLETE:
  df = df[df['subset'] != 'complete']

In [ ]:
model = "FCNN"
metrics = ["accuracy", "f1"]

colors = {
    "complete": "black",
    str(Max_match_0) + ' - TM (Max) & TMP (Max)': 'limegreen',
    str(Max_unmatch_0) + ' - TU (Max)': "red",
    str(Min_match_0) + ' - TM (Min)': "blue",
    str(Min_unmatch_0) + ' - TU (Min)': "orange",
    str(Min_mixup_0) + ' - TMP (Min)': 'teal'

}

for metric in metrics:

    plt.figure(figsize=(10, 6))

    for subset in sorted(df["subset"].unique()):

        values = df[
            (df["model"] == model) &
            (df["subset"] == subset)
        ][metric]

        mu = values.mean()
        sigma = values.std()

        if sigma == 0 or len(values) == 0:
            continue

        x = np.linspace(
            mu - 4 * sigma,
            mu + 4 * sigma,
            500
        )

        is_special = subset in colors

        color = colors.get(subset, "gray")
        linewidth = 3 if is_special else 1
        alpha = 1 if is_special else 0.5
        linestyle = "-" if is_special else "--"
        zorder = 3 if is_special else 1

        plt.plot(
            x,
            norm.pdf(x, mu, sigma),
            color=color,
            linewidth=linewidth,
            alpha=alpha,
            linestyle=linestyle,
            zorder=zorder,
            label=subset
        )

    plt.legend()
    plt.xlabel(metric.upper())
    plt.ylabel("Density")
    plt.title(f"SNIPS 384 - Normal Distribution ({metric})")
    plt.show()